# ViSpatialQA — Thí nghiệm E3: LLM chuyển văn bản thành JSON, S suy luận (nhánh LP)

Mục tiêu: cải thiện **Human**, nhất là **FB** (lượng từ, phủ định), cùng CO và YN. Lý do chọn hướng này: `output/base/khao_sat_paper.md`, mục E3.

- **Cách làm:**
  - Qwen3-32B đọc mỗi story và chép lại thành JSON: vật thể, khối, các quan hệ được nói rõ, chạm cạnh. Mỗi story lấy `n_world` mẫu.
  - Mỗi câu hỏi được chuyển thành dạng logic, lấy `n_form` mẫu.
  - Bộ suy luận của S (đảo chiều, bắc cầu, kế thừa theo khối) trả lời từng cặp (world, dạng logic), rồi bỏ phiếu.
  - Không có cặp nào dùng được thì câu đó **abstain**, và ensemble dùng nhánh khác.
- **Kiểm tra trước (oracle, nếu LLM chép đúng hoàn toàn):** trên 3 story Human train tự chú thích tay, đúng 39/48 câu. Thêm quy ước "khác khối là xa" thì đúng 42/48 (≈0.875). Cao nhất ở FB: 12/12.
- **Chọn quy ước trên Human train:** LP không train, nên toàn bộ Human train (trừ story 0, là ví dụ trong prompt) được dùng để chọn quy ước trả lời. Tập này gấp khoảng 4 lần dev, nên đỡ nhiễu. Việc chọn chạy lại trên CPU từ file parses, **không cần chạy lại GPU**.
- **So sánh cuối:** tổ hợp hiện tại (E2: `F=F2C,F1 S=S L=LCOT`) so với cùng tổ hợp thêm nhánh `P=LP`, bằng cv 5-fold trên dev như trước.

**Cần có trước:** dự đoán nền F1, F2C, S, LCOT trên Drive (`outputs/predictions_e1e2`, do notebook E1/E2 tạo).

**Thời gian ước tính** (H100/Blackwell): Human train + dev + test ≈ 50 story × 4 world và ≈ 740 câu × 4 form, khoảng 30–60 phút.

## 1. CONFIG

In [ ]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/e3-llm-parse",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    HF_USER="",                                   # để trống = lấy từ token
    USE_WANDB=True, WANDB_PROJECT="vispatialqa-vlsp2026",
    BASE_SOURCES=["F1", "F2", "F2C", "S", "LCOT"],  # lấy từ Drive outputs/predictions_e1e2
    LP=dict(model="Qwen/Qwen3-32B-FP8", fallback="Qwen/Qwen3-32B", n_world=4, n_form=4,
            max_tokens=8192, max_model_len=16384, max_num_seqs=64, gpu_memory_utilization=0.90,
            think_forms=True,                     # False = chuyển câu hỏi không suy nghĩ (nhanh hơn)
            splits=["train", "dev", "test"]),
    SMOKE_TEST=True,                              # chạy thử 16 câu Human dev trước khi chạy thật
    USE_INDIFINITE=False, HUMAN_YN_DK="keep",
    SUBMISSION_NAMES=dict(human="human_submission.json", auto="auto_submission.json"),
    SEED=42,
)

## 2. Chuẩn bị: code nhánh E3, thư viện, Drive, dữ liệu, dự đoán nền, HF/W&B

In [ ]:
import codecs, json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

C = CONFIG
WORK = Path(C["WORK_DIR"])
OUT = WORK / "outputs"
PRED = OUT / "predictions_e3"        # dự đoán của notebook này + bản sao dự đoán nền
DRIVE_OUT = Path(C["DRIVE_BACKUP_DIR"])
os.environ["VLLM_USE_DEEP_GEMM"] = "0"            # FP8 JIT cần NVCC >= 12.9
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # sampler JIT cần CUDA >= 12.9 trên GPU SM 12.x
START = time.time()


def sh(cmd, check=True):
    """Chạy lệnh (list hoặc str) trong WORK_DIR, in output trực tiếp (giữ thanh tiến trình)."""
    if isinstance(cmd, (list, tuple)):
        cmd = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {cmd}", flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, shell=True, cwd=WORK if WORK.exists() else None, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while chunk := os.read(process.stdout.fileno(), 4096):
        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
    code = process.wait()
    print(f"\n[exit {code}, {(time.time() - started) / 60:.1f} phút | tổng {(time.time() - START) / 60:.1f} phút]")
    if check and code:
        raise RuntimeError(f"Lệnh lỗi (exit {code}): {cmd}")
    return code


def gpu(title=""):
    print(f"--- nvidia-smi {title}")
    subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", shell=True)


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def split_file(dataset, split):
    return {"train": f"Data/splits/{dataset}_train.json", "dev": f"Data/splits/{dataset}_dev.json",
            "test": f"Data/{dataset}_public_test.json"}[split]


def jobs(source, pairs):
    """--job arguments writing into outputs/predictions_e1e2/<source>/<dataset>_<split>.jsonl."""
    out = []
    for dataset, split in pairs:
        out += ["--job", f"{dataset},{split_file(dataset, split)},{PRED / source / f'{dataset}_{split}.jsonl'}"]
    return out


def to_drive(folder):
    if Path("/content/drive/MyDrive").exists() and (OUT / folder).exists():
        shutil.copytree(OUT / folder, DRIVE_OUT / folder, dirs_exist_ok=True)
        print(f"Đã sao lưu {OUT / folder} -> {DRIVE_OUT / folder}")


# Drive, code, thư viện
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
if subprocess.run([sys.executable, "-c", "import vllm, peft"], capture_output=True).returncode != 0:
    sh("pip install -q vllm")
    sh("pip install -q peft datasets accelerate huggingface_hub wandb")
sh("pip uninstall -y -q torchaudio torchao", check=False)   # bản Colab không tương thích torch của vllm

# Dữ liệu + split
for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/auto_dev.json").exists():
    sh("python make_splits.py --from-manifest")

# Resume sau khi Colab ngắt (predictions_e3), rồi dự đoán nền của notebook E1/E2 (predictions_e1e2)
for folder, sources in (("predictions_e3", None), ("predictions_e1e2", C["BASE_SOURCES"])):
    root = DRIVE_OUT / folder
    for path in root.rglob("*.jsonl") if root.exists() else []:
        target = PRED / path.relative_to(root)
        if (not target.exists() and not path.name.endswith(".raw.jsonl")
                and (sources is None or path.parent.name in sources)):
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
PRED.mkdir(parents=True, exist_ok=True)

# HF + W&B
os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN"))
if not os.environ.get("HF_USER"):
    os.environ["HF_USER"] = C.get("HF_USER") or ""
if not os.environ["HF_USER"] and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    os.environ["HF_USER"] = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
if C["USE_WANDB"] and not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
os.environ.setdefault("WANDB_PROJECT", C["WANDB_PROJECT"])
os.environ["WANDB_LOG_MODEL"] = "false"
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

In [ ]:
missing = [f"{s}/human_dev.jsonl" for s in ("F1", "F2C", "S", "LCOT") if not (PRED / s / "human_dev.jsonl").exists()]
if "S/human_dev.jsonl" in missing or not (PRED / "S/auto_dev.jsonl").exists():
    sh(["python", "solve_symbolic.py", "--run-name", "E3-base-S"] + sum(
        (["--job", f"{d},{split_file(d, s)},{PRED / 'S' / f'{d}_{s}.jsonl'}"] for d in ("auto", "human") for s in ("dev", "test")), []))
    missing = [m for m in missing if not m.startswith("S/")]
assert not missing, f"Thiếu dự đoán nền {missing}: chạy notebook run_colab_e1e2.ipynb trước (nó sao lưu vào Drive)."
print("Dự đoán nền:", sorted(str(p.relative_to(PRED)) for p in PRED.rglob("*.jsonl") if ".raw." not in p.name))

## 3. Chạy LP (Qwen3-32B chép story + câu hỏi thành JSON)
Chạy thử 16 câu trước (in số JSON không đọc được), sau đó chạy thật trên Human train/dev/test trong một lần nạp mô hình.

In [ ]:
LP = C["LP"]


def run_lp(pairs, extra, source="LP"):
    base = ["python", "llm_parse.py", "--source", source, "--n-world", LP["n_world"], "--n-form", LP["n_form"],
            "--max-tokens", LP["max_tokens"], "--max-model-len", LP["max_model_len"], "--max-num-seqs", LP["max_num_seqs"],
            "--gpu-memory-utilization", LP["gpu_memory_utilization"], "--seed", C["SEED"], "--fallback-model", "",
            *jobs(source, pairs), *extra] + ([] if LP["think_forms"] else ["--no-think-forms"])
    if sh(base + ["--model-name", LP["model"]], check=False) != 0:
        print(f"⚠️ {LP['model']} không chạy được -> thử {LP['fallback']}")
        sh(base + ["--model-name", LP["fallback"]])


gpu("trước LP")
if C["SMOKE_TEST"]:
    run_lp([("human", "dev")], ["--limit", 16, "--run-name", "E3-LP-smoke"], source="LP_smoke")
    worlds = [json.loads(l) for l in open(PRED / "LP_smoke/human_dev.parses.jsonl", encoding="utf-8")]
    print("Mẫu world đầu tiên:", json.dumps(next(w for w in worlds if w["kind"] == "world")["worlds"][0], ensure_ascii=False)[:1500])
run_lp([("human", s) for s in LP["splits"]], ["--run-name", "E3-LP-human"])
gpu("sau LP")
to_drive("predictions_e3")

## 4. Chọn quy ước trả lời trên Human train (CPU, từ file parses)
Thử mọi tổ hợp công tắc: YN đóng/mở, FR nhiều cặp tất cả/bất kỳ, FB toàn cục/trong khối, "xa" lan theo hướng, "khác khối là xa". Chọn tổ hợp tốt nhất trên **Human train** (coi abstain là sai), rồi áp dụng cho dev/test → nguồn **LPT**. Dev chỉ dùng để xem, không dùng để chọn.

In [ ]:
(OUT / "e3").mkdir(parents=True, exist_ok=True)
sh(["python", "solve_parsed.py", "--parses", PRED / "LP/human_train.parses.jsonl", "--input", split_file("human", "train"),
    "--grid", "--best-json", OUT / "e3/best_convention.json"])
best = json.loads((OUT / "e3/best_convention.json").read_text())
print("Quy ước chọn trên train:", best["settings"])
settings = sum((["--set", s] for s in best["settings"]), [])
for split in ("dev", "test"):
    sh(["python", "solve_parsed.py", "--parses", PRED / f"LP/human_{split}.parses.jsonl", "--input", split_file("human", split),
        "--source", "LPT", "--output", PRED / f"LPT/human_{split}.jsonl", *settings])
to_drive("predictions_e3")

## 5. So sánh: tổ hợp hiện tại (E2) và thêm nhánh LP
`P=LPT` là nhánh mới. Ensemble tune trọng số theo từng loại câu (cv 5-fold theo story trên dev như trước), và LP abstain thì dùng nhánh khác.

In [ ]:
from IPython.display import Markdown, display

VARIANTS = {
    "E2 (hiện tại)": "F=F2C,F1 S=S L=LCOT",
    "E2 + LP": "F=F2C,F1 S=S L=LCOT P=LPT",
    "E2 + LP (không L)": "F=F2C,F1 S=S P=LPT",
}
names = ",".join(f"{d}={n}" for d, n in C["SUBMISSION_NAMES"].items())
summary = ["| Biến thể | Tổ hợp tốt nhất | Human cv | Auto cv | **Final cv** | Human YN | Human FR | Human FB | Human CO |",
           "|---|---|---|---|---|---|---|---|---|"]
for variant, branches in VARIANTS.items():
    out_dir = OUT / "e3" / variant.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")
    sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", out_dir, "--branches", branches,
        "--human-yn-dk", C["HUMAN_YN_DK"], "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite",
        "--submission-names", names, "--run-name", f"E3-compare-{out_dir.name}"])
    best_combo = json.loads((out_dir / "best.json").read_text())
    result = json.loads((out_dir / "comparison.json").read_text())[best_combo["combination"]]
    human, auto = result.get("human"), result.get("auto")
    fmt = lambda v: f"{v:.3f}" if v is not None else "n/a"
    per = human["cv"] if human else {}
    summary.append(f"| {variant} | {best_combo['combination']} | {fmt(human and human['cv_mean'])} | "
                   f"{fmt(auto and auto['cv_mean'])} | **{best_combo['final_cv']:.4f}** | " + " | ".join(
                       fmt(per.get(t, {}).get("accuracy" if t in ("YN", "CO") else "exact_match")) for t in ("YN", "FR", "FB", "CO")) + " |")
text = "\n".join(summary)
display(Markdown(text))
(OUT / "e3/summary.md").write_text(text + f"\n\nQuy ước LP (chọn trên Human train): {best['settings']}\n", encoding="utf-8")
to_drive("e3")
print(f"Xong sau {(time.time() - START) / 60:.1f} phút. Bảng: {OUT / 'e3/summary.md'}")